# 1、关于对话模型中消息（message）的使用

标准的对话模型的调用过程：

invoke()的输入可以是多种类型，典型的类型有：① 字符串类型 ② 消息列表

invoke()的输出类型：BaseMessage的子类：AIMessage

In [ ]:
from langchain_openai import ChatOpenAI
import os
import dotenv

# 前提：加载配置文件
dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY1")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

# 1、获取对话模型
chat_model = ChatOpenAI(
    # api_key=,
    # base_url=,
    model_name="qwen2.5-coder:1.5b",
)

# 2、调用对话模型
response = chat_model.invoke("帮我解释一下什么是langchain?")

# 3、处理响应数据
# print(response)
print(response.content)
print(type(response))  #<class 'langchain_core.messages.ai.AIMessage'>

关于消息：

有典型的三类消息：SystemMessage \ HumanMessage \ AIMessage

举例1：


In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_message = SystemMessage(content="你是一个英语教学方向的专家")
human_message = HumanMessage(content="帮我制定一个英语六级学习的计划")

messages = [system_message, human_message]

print(messages)

使用大模型，调用消息列表

In [ ]:
response = chat_model.invoke(messages)
print(type(response))
print(response.content)

举例2：

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

system_message = SystemMessage(
    content="你是一个英语教学方向的专家",
    additional_kwargs={"tool":"invoke_func1"}
)
human_message = HumanMessage(content="帮我制定一个英语六级学习的计划")

messages = [system_message, human_message]

print(messages)

举例3:ChatMessage平时我们使用的不多，了解一下即可

In [ ]:
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ChatMessage
)

# 创建不同类型的消息
system_message = SystemMessage(content="你是一个专业的数据科学家")
human_message = HumanMessage(content="解释一下随机森林算法")
ai_message = AIMessage(content="随机森林是一种集成学习方法...")
custom_message = ChatMessage(role="analyst", content="补充一点关于超参数调优的信息")

print(system_message.content)
print(human_message.content)
print(ai_message.content)
print(custom_message.content)

# 2、关于多轮对话与上下文记忆

前提：

In [ ]:
import os
import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
	model="qwen2.5-coder:1.5b"
)

测试1：大模型本身是没有上下文记忆能力的



In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")

messages = [sys_message, human_message]

#调用大模型，传入messages
response = chat_model.invoke(messages)
# print(response.content)


response1 = chat_model.invoke("你叫什么名字？")
print(response1.content)

测试2：

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")
human_message1 = HumanMessage(content="你叫什么名字？")

messages = [sys_message, human_message,human_message1]

#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)


测试3：

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")

sys_message1 = SystemMessage(
    content="我可以做很多事情，有需要就找我吧",
)

human_message1 = HumanMessage(content="你叫什么名字？")

messages = [sys_message, human_message,sys_message1,human_message1]

#调用大模型，传入messages
response = chat_model.invoke(messages)
print(response.content)


测试4：

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

# 第1组
sys_message = SystemMessage(
    content="我是一个人工智能的助手，我的名字叫小智",
)
human_message = HumanMessage(content="猫王是一只猫吗？")

messages = [sys_message, human_message]

# 第2组
sys_message1 = SystemMessage(
    content="我可以做很多事情，有需要就找我吧",
)

human_message1 = HumanMessage(content="你叫什么名字？")

messages1 = [sys_message1,human_message1]

#调用大模型，传入messages
response = chat_model.invoke(messages)
# print(response.content)

response = chat_model.invoke(messages1)
print(response.content)

测试5：

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="AI"),
    HumanMessage(content="你叫什么名字"),
]

messages1 = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="很高兴认识你"),
    AIMessage(content="我也很高兴认识你"),
    HumanMessage(content="你叫什么名字"),
]

messages2 = [
    SystemMessage(content="我是一个人工智能助手，我的名字叫小智"),
    HumanMessage(content="人工智能英文怎么说？"),
    AIMessage(content="AI"),
    HumanMessage(content="你叫什么名字"),
]

chat_model.invoke(messages2)

# 3、关于模型调用的方法的说明

invoke() / stream()

batch():批量的调用

ainvoke() / astream() / abatch():异步方法的调用

举例1：体会invoke()阻塞式的调用

In [ ]:
import os
import dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

#初始化大模型
chat_model = ChatOpenAI(model="qwen2.5-coder:1.5b")

# 创建消息
messages = [HumanMessage(content="你好，请介绍一下自己")]

# 非流式调用LLM获取响应
response = chat_model.invoke(messages)

# 打印响应内容
print(response)

举例2：流式的演示

In [ ]:
import os
import dotenv
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 初始化大模型
chat_model = ChatOpenAI(model="qwen2.5-coder:1.5b",
                        streaming=True  # 启用流式输出
                        )

# 创建消息
messages = [HumanMessage(content="你好，请介绍一下自己")]

# 流式调用LLM获取响应
print("开始流式输出：")
for chunk in chat_model.stream(messages):
    # 逐个打印内容块
    print(chunk.content, end="", flush=True) # 刷新缓冲区 (无换行符，缓冲区未刷新，内容可能不会立即显示)

print("\n流式输出结束")

举例3：使用batch，测试批量调用

In [ ]:
import os
import dotenv
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()

os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY1")
os.environ['OPENAI_BASE_URL'] = os.getenv("OPENAI_BASE_URL")

# 初始化大模型
chat_model = ChatOpenAI(model="qwen2.5-coder:1.5b")

messages1 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
             HumanMessage(content="请帮我介绍一下什么是机器学习"), ]

messages2 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
             HumanMessage(content="请帮我介绍一下什么是AIGC"), ]

messages3 = [SystemMessage(content="你是一位乐于助人的智能小助手"),
             HumanMessage(content="请帮我介绍一下什么是大模型技术"), ]

messages = [messages1, messages2, messages3]

# 调用batch
response = chat_model.batch(messages)

print(response)


举例4：关于同步和异步方法的调用

体会1：

In [ ]:
import time

def call_model():
    # 模拟同步API调用
    print("开始调用模型...")
    time.sleep(5)  # 模拟调用等待,单位：秒
    print("模型调用完成。")

def perform_other_tasks():
    # 模拟执行其他任务
    for i in range(5):
        print(f"执行其他任务 {i + 1}")
        time.sleep(1)  # 单位：秒

def main():
    start_time = time.time()
    call_model()
    perform_other_tasks()
    end_time = time.time()
    total_time = end_time - start_time
    return f"总共耗时：{total_time}秒"

# 运行同步任务并打印完成时间
main_time = main()
print(main_time)

体会2：

In [ ]:
import asyncio
import time

async def async_call(llm):
    await asyncio.sleep(5)  # 模拟异步操作
    print("异步调用完成")

async def perform_other_tasks():
    await asyncio.sleep(5)  # 模拟异步操作
    print("其他任务完成")

async def run_async_tasks():
    start_time = time.time()
    await asyncio.gather(
        async_call(None),  # 示例调用，使用None模拟LLM对象
        perform_other_tasks()
    )
    end_time = time.time()
    return f"总共耗时：{end_time - start_time}秒"

# # 正确运行异步任务的方式
# if __name__ == "__main__":
#     # 使用 asyncio.run() 来启动异步程序
#     result = asyncio.run(run_async_tasks())
#     print(result)


# 在 Jupyter 单元格中直接调用
result = await run_async_tasks()
print(result)